In [1]:
!/usr/local/cuda/bin/nvcc --version
!nvidia-smi

The system cannot find the path specified.


Thu Jul 23 10:04:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.62                 KMD Version: 610.62        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070      WDDM  |   00000000:01:00.0  On |                  N/A |
| 39%   46C    P8             17W /  270W |    1654MiB /   8192MiB |      6%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu129
!pip3 install -U "ray[default]"

In [3]:
import numpy as np
import json
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline
import shutil
import os
import os.path as osp
import glob
import random
from tqdm import tqdm
import datetime

import torch
import torchvision
from torchvision import datasets, models, transforms
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.autograd import Variable

In [4]:
transform = transforms.Compose([
    transforms.Resize(800),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

# Dataset setup

In [ ]:
!pip install python-dotenv
!pip install -U ipywidgets
!python3 -m pip install --upgrade setuptools pip wheel
!python3 -m pip install nvidia-pyindex
!python3 -m pip install nvidia-cuda-runtime-cu12

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv() # load paths from .env
# Directory containing the images
train_images_path = os.getenv("TRAIN_IMAGES_PATH")
train_masks_path = os.getenv("TRAIN_MASKS_PATH")
val_images_path = os.getenv("VAL_IMAGES_PATH")
val_masks_path = os.getenv("VAL_MASKS_PATH")
print('Training Data')
for class_name in os.listdir(train_images_path):
    print(len(os.listdir(os.path.join(train_images_path))))
print('\nValidation Data')
for class_name in os.listdir(val_images_path):
    print(len(os.listdir(os.path.join(val_images_path))))

Adapt this into a segmentation model for PyTorch, previously worked on object detection with PyTorch.
We should also look into seeing how we would apply a ViT (Vision transformer)

In [12]:
# Start a basic local cluster head node
# !ray start --head

# Optional: Start with specific resource allocations
!ray start --head --num-cpus=8 --num-gpus=1

2026-07-23 10:08:44,451	INFO usage_lib.py:474 -- Usage stats collection is enabled by default without user confirmation because this terminal is detected to be non-interactive. To disable this, add `--disable-usage-stats` to the command that starts the cluster, or run the following command: `ray disable-usage-stats` before starting the cluster. See https://docs.ray.io/en/master/cluster/usage-stats.html for more details.
2026-07-23 10:08:44,451	INFO scripts.py:1042 -- Local node IP: 127.0.0.1
2026-07-23 10:08:49,935	SUCC scripts.py:1081 -- --------------------
2026-07-23 10:08:49,935	SUCC scripts.py:1082 -- Ray runtime started.
2026-07-23 10:08:49,935	SUCC scripts.py:1083 -- --------------------
2026-07-23 10:08:49,935	INFO scripts.py:1085 -- Next steps
2026-07-23 10:08:49,935	INFO cli_output_helpers.py:16 -- Note: The following commands are intended for use on
2026-07-23 10:08:49,935	INFO cli_output_helpers.py:17 -- the head node or within the cluster network.
2026-07-23 10:08:49,935	I

In [ ]:
from dotenv import load_dotenv
import ray
import gc

if (ray.is_initialized()):
    ray.shutdown()
    gc.collect()

ray.init(address="auto")
load_dotenv() # load paths from .env
# Directory containing the images
train_images = ray.data.read_images(paths=os.getenv("TRAIN_IMAGES_PATH"), file_extensions=["jpg"])
train_masks = ray.data.read_images(paths=os.getenv("TRAIN_MASKS_PATH"), file_extensions=["png"])
val_images = ray.data.read_images(paths=os.getenv("VAL_IMAGES_PATH"), file_extensions=["jpg"])
val_masks = ray.data.read_images(paths=os.getenv("VAL_MASKS_PATH"), file_extensions=["png"])

print('Training Data')
print("train_images:\n", train_images.schema())
print("train_masks:\n", train_masks.schema())
print("val_images:\n", val_images.schema())
print("val_masks:\n" ,val_masks.schema())

In [ ]:
!ray list actors

In [ ]:
import gc

ray.shutdown()
gc.collect()

In [ ]:
from ray.util.queue import Queue
q = Queue()

# Get total cluster resources
print(ray.cluster_resources())

# Get current available (idle) cluster resources
print(ray.available_resources())


print(q.empty())

In [ ]:
# If cluster resources do not clear up, run this command and start over
!ray stop --force